### IMPLEMENTING GPT MODEL FROM SCRATCH TO GENERATE TEXT

In [1]:
import torch

In [2]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # Vocabulary size of the model
    "context_length": 1024,  # Maximum context length for the model
    "emb_dim": 768,          # Embedding dimension of the model
    "n_heads": 12,           # Number of attention heads in the model
    "n_layers": 12,          # Number of transformer layers in the model
    "drop_rate": 0.1,        # Dropout rate for regularization
    "qkv_bias": True         # Whether to include bias terms in the query, key, and value projections
}

## GPT Architecture Part 1: Dummy GPT Model Class

#### Step 1: Use a placeholder for Transformer Block

#### Step 2: Use a placeholder for LayerNorm

In [12]:
import torch
import torch.nn as nn


class DummyGPTmodel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tok_emb = nn.Embedding(config["vocab_size"], config["emb_dim"])
        self.pos_emb = nn.Embedding(config["context_length"], config["emb_dim"])
        self.drop_emb = nn.Dropout(config["drop_rate"])

        # Use a placeholder for transformer block
        self.trf_blocks = nn.Sequential(
            *[DummyTransformerBlock(config) for _ in range(config["n_layers"])])

        # Use a placeholder for LayerNorm
        self.final_norm = DummyLayerNorm(config["emb_dim"])
        self.out_head = nn.Linear(config["emb_dim"], config["vocab_size"], bias=False)


    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_emb = self.tok_emb(in_idx)  # Token embeddings
        pos_emb = self.pos_emb(torch.arange(seq_len, device=in_idx.device))  # Positional embeddings
        x = tok_emb + pos_emb  # Combine token and positional embeddings
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [13]:
class DummyTransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        # placeholder

    def forward(self, x):
        
        return x
    

class DummyLayerNorm(nn.Module):
    def __init__(self, normalized_shape, eps=1e-5):
        super().__init__()
        # placeholder

    def forward(self, x):
        return x

### STEP 1: Tokenization

In [14]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
batch = []
txt1 = "Every effort moves you"
txt2 = "Every day holds a"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[6109, 3626, 6100,  345],
        [6109, 1110, 6622,  257]])


## STEP 2: Create an instance of DummyGPTModel

In [15]:
torch.manual_seed(123)
model = DummyGPTmodel(GPT_CONFIG_124M)
logits = model(batch)
print(logits.shape)  # Should be (batch_size, seq_len, vocab_size)
print(logits)  # Print the logits for inspection

torch.Size([2, 4, 50257])
tensor([[[-1.2034,  0.3201, -0.7130,  ..., -1.5548, -0.2390, -0.4667],
         [-0.1192,  0.4539, -0.4432,  ...,  0.2392,  1.3469,  1.2430],
         [ 0.5307,  1.6720, -0.4695,  ...,  1.1966,  0.0111,  0.5835],
         [ 0.0139,  1.6754, -0.3388,  ...,  1.1586, -0.0435, -1.0400]],

        [[-1.0908,  0.1798, -0.9484,  ..., -1.6047,  0.2439, -0.4530],
         [-0.7860,  0.5581, -0.0610,  ...,  0.4835, -0.0077,  1.6621],
         [ 0.3567,  1.2698, -0.6398,  ..., -0.0162, -0.1296,  0.3717],
         [-0.2407, -0.7349, -0.5102,  ...,  2.0057, -0.3694,  0.1814]]],
       grad_fn=<UnsafeViewBackward0>)


- The output has 2 rows correspoding to 2 text samples.
- Each text sample consits of 4 tokens; each token is a 50,257-dimensional vector, which matches the size of tokenizer's vocabulary

- The embedding has 50,257 dimensions because each of these dimensions refers to a unique token in the vocabulary. 
- At the end, these 50,257-dimensional vectors will be converted back into tokenIDs, which we can then decode into words.

## GPT2 Architecture Part 2: LAYER NORMALIZATION